# LAYA factual hallucination judgments on saved TriviaQA outputs (GPU)

This separate run reuses the saved Qwen predictions and BLEURT-20 scores from `nvthaai/sala-g14-6-source-triviaqa-version-6-artifacts`. LAYA sees only `question` and `assistant_response`. It returns hallucination probability, primary issue type, and severity. Set the accelerator to **GPU** and enable Internet.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!pip install -q "git+https://github.com/NandhaKishorM/laya.git"

In [ ]:
from pathlib import Path
import subprocess, sys

SOURCE_NAME = "sala_qwen25_g14_6src_triviaqa_v1"
artifact_dirs = [
    path for path in Path("/kaggle/input").rglob(SOURCE_NAME)
    if (path / "generation/triviaqa/rows.jsonl").is_file()
    and (path / "bleurt/triviaqa/scores.jsonl").is_file()
]
if len(artifact_dirs) != 1:
    raise RuntimeError(f"Expected one completed artifact directory, found: {artifact_dirs}")
ARTIFACT_DIR = artifact_dirs[0]
OUTPUT_DIR = Path("/kaggle/working/laya_triviaqa_gpu_multijudge")
print(f"Using saved TriviaQA artifacts: {ARTIFACT_DIR}")

command = [
    sys.executable, "/kaggle/working/SALA/laya_triviaqa.py",
    "--artifact-dir", str(ARTIFACT_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--device", "cuda",
    "--batch-size", "8",
    "--threshold", "0.5",
]
subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
results = pd.read_csv(OUTPUT_DIR / "results.csv")
display(results.head())
print(json.dumps(json.loads((OUTPUT_DIR / "summary.json").read_text()), indent=2))
print(f"Saved {len(results):,} rows to {OUTPUT_DIR}")